# Headroom compression playbook

Two compressions from the Module 1 RAG lab (`hw1.ipynb`), measured **before / after** on the same data:

1. **RAG context blob** — the 5 retrieved chunks concatenated into the prompt (`RAGBase.build_context`). One-shot.
2. **Agentic search-tool outputs** — what `search()` returns each turn, which the agent **re-sends every loop**.

Compression engine: Headroom `compress()` backed by the Kompress-base text model (the `[ml]` extra).

**What we found while wiring this up (drives the design below):**
- `compress()` **protects user/system messages by default** → opt in with `compress_user_messages=True`.
- JSON tool payloads route to `SmartCrusher` (structural only). Our chunks are **prose + code**, so the win comes from the Kompress **text** path; we compress the *rendered text block* each stage feeds the model.
- These lessons are **code-heavy**, and Headroom **protects recent code by default** (`router:protected:recent_code`) → ~0% out of the box (the safe default). Turning off the exposed protections (`protect_recent=0`, `protect_analysis_context=False`, and opt-in user/system compression) lets it compress: it switches to a **mixed** router that shortens prose and trims low-value code. Because that can drop exact code lines, we **verify the answer end-to-end** rather than trusting the token count.

**Environment notes** (this machine sits behind a TLS-inspecting proxy):
- `truststore.inject_into_ssl()` makes Python trust the corporate CA so the model can download.
- `HF_HUB_DISABLE_XET=1` forces the pure-Python HF download path (the Rust xet downloader hangs on the proxy CA).
- Put `HF_TOKEN` in `.env` (loaded by `load_dotenv()` in the setup cell) to avoid unauthenticated Hub warnings.
- After Kompress is cached (~0.6 GB under `~/.cache/huggingface/`), set `HF_HUB_OFFLINE=1` in `.env` to use the cache only.

In [1]:
import os

# --- make HF model fetch work behind a TLS-inspecting proxy ---
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import truststore

truststore.inject_into_ssl()

import transformers

transformers.logging.set_verbosity_error()

from dotenv import load_dotenv

load_dotenv()

from headroom import compress

# Tokenizer/model name Headroom uses for counting. The lab's LLM is separate
# (served via OpenRouter); this only drives Headroom's token accounting.
HR_MODEL = "gpt-4o"

## Load the same data as the lab

Same source, chunking, and index as `hw1.ipynb` (chunk `size=2000`, `step=1000`).

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [f.parse() for f in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

len(documents), len(chunks)

(72, 295)

## Compression helper

One primitive for both scenarios: take the **text block** that is about to be sent to
the model and compress it. We opt into user-message compression and let Kompress-base
do the prose work. `CompressResult` already reports tokens before/after.

### Compression knobs — what controls what

Headroom works on **tokens, not characters**. There is no "number of chars to compress"
setting — the `[:400]` you see below is only a *print preview* of the result and has zero
effect on compression. The real knobs (all passed through `compress_block`) are:

| Knob | Default here | Effect | Push it and... |
| --- | --- | --- | --- |
| `target_ratio` | `0.5` | A **hint/cap**, not a hard target. The router still decides what's achievable | Lower (e.g. `0.2`) → **no change for our prose+code** (see stress test below); the router caps it at ~0.52 either way |
| `min_tokens_to_compress` | `50` | Skip blocks smaller than N tokens | Higher → leaves short chunks untouched (safer, less savings) |
| `protect_recent` | `0` | Protects the last N messages that look like code | `>0` → keeps code verbatim (our default `0` is what unlocked ~15%) |
| `protect_analysis_context` | `False` | Protects code when "analysis intent" is detected | `True` → won't compress code-ish text |
| `compress_user_messages` | `True` | Whether user-role text is eligible at all | `False` → user content never compressed |
| `compress_system_messages` | `True` | Whether system/developer text is eligible | `False` → system content never compressed |

> **Gotcha:** the router only compresses once the Kompress model is initialized. The first
> `compress()` on a cold model can return `router:noop` (0%). The warmup call in the helper
> cell below avoids that, so later measurements are stable.

**What gets impacted as you compress harder:** token savings go up, but exact strings
(`while True`, `break`) can disappear, which raises the risk on code-detail questions.
Headroom also self-reverts a block if compressing it would *inflate* tokens
(`Optimization inflated tokens ...; reverting`). That's why every scenario here ends with
an **end-to-end answer check** rather than trusting the token count alone.

In [3]:
def compress_block(
    text,
    target_ratio=0.5,
    protect_recent=0,
    protect_analysis_context=False,
):
    """Compress a single text block the way it would be sent to the LLM.

    This is the minimal "all exposed protections off" setup for this lab:
    - opt into user/system compression
    - disable recent-code protection
    - disable analysis-context protection
    """
    return compress(
        [{"role": "user", "content": text}],
        model=HR_MODEL,
        compress_user_messages=True,
        compress_system_messages=True,
        min_tokens_to_compress=50,
        target_ratio=target_ratio,
        protect_recent=protect_recent,
        protect_analysis_context=protect_analysis_context,
    )


def render_results(search_results):
    """Same shape as RAGBase.build_context: filename + content per hit."""
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


# Warm up the model once so later cells time cleanly (first load ~60s).
_warm = compress_block("warm up the kompress model " * 30)
print("model ready — warmup:", _warm.tokens_before, "->", _warm.tokens_after, _warm.transforms_applied)

model ready — warmup: 188 -> 137 ['router:text:0.50']


## Scenario A — RAG context blob

The one-shot RAG path retrieves 5 chunks and concatenates them into the prompt. We
compress that blob and measure the token delta.

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"

results = chunk_index.search(query, num_results=5)
context = render_results(results)

# Default behaviour after opting into user compression: Headroom protects the code.
res_default = compress_block(
    context,
    target_ratio=0.5,
    protect_recent=4,
    protect_analysis_context=True,
)
# Protections off: allow the mixed prose+code router to compress.
resA = compress_block(context, target_ratio=0.5)

print("RAG context blob")
print(f"  default protections: {res_default.tokens_before} -> {res_default.tokens_after} "
      f"({res_default.compression_ratio:.0%})  {res_default.transforms_applied}")
print(f"  exposed protections off: {resA.tokens_before} -> {resA.tokens_after} "
      f"({resA.compression_ratio:.0%})  {resA.transforms_applied}")
print("\n--- compressed context (first 400 chars) ---")
print(resA.messages[0]["content"][:400])

RAG context blob
  default protections: 2224 -> 2224 (0%)  ['router:protected:recent_code']
  exposed protections off: 2224 -> 1894 (15%)  ['router:mixed:0.52']

--- compressed context (first 400 chars) ---
01-agentic-rag/lessons/14-agentic-loop.md loop. loop keeps calling response without function calls. also keep iteration counter see round-trips happened.
[32 items compressed to 16. Retrieve more: hash=8e320d1076ecb0a7fed97e5f]

```python
1 print(f"iteration False response openai_client.responses.create( model="gpt-5.4-mini", input=messages, tools=[search_tool], messages.extend(response.output) in


## Placement test — before indexing vs after retrieval

The reason to keep the index on raw chunks is simple: `minsearch` is lexical. If we compress before indexing, Headroom may rewrite or remove exact tokens that are useful search keys (`while True`, `break`, etc.).

To keep this precise and fast, we test the actual 5 chunks retrieved for the main query:

- build one tiny index over the raw retrieved chunks;
- build one tiny index over the same chunks after Headroom compression;
- search both with code-sensitive diagnostic queries.

This isolates the effect of **compression before indexing** without re-compressing the whole corpus.

In [5]:
candidate_chunks = [{**doc, "chunk_id": str(i)} for i, doc in enumerate(results)]
compressed_candidate_chunks = []

for doc in candidate_chunks:
    compressed = compress_block(doc["content"], target_ratio=0.5).messages[0]["content"]
    compressed_candidate_chunks.append({**doc, "content": compressed})

raw_candidate_index = Index(text_fields=["content"], keyword_fields=["filename", "chunk_id"])
raw_candidate_index.fit(candidate_chunks)

compressed_candidate_index = Index(text_fields=["content"], keyword_fields=["filename", "chunk_id"])
compressed_candidate_index.fit(compressed_candidate_chunks)

terms = ["while True", "break", "has_function_calls", "function_call", "messages.extend"]
print("Exact term preservation across the retrieved chunks:")
for term in terms:
    raw_hits = sum(term in doc["content"] for doc in candidate_chunks)
    comp_hits = sum(term in doc["content"] for doc in compressed_candidate_chunks)
    print(f"  {term:20} raw={raw_hits} compressed-before-index={comp_hits}")

diagnostic_queries = [
    "while True break",
    "has_function_calls False break",
    "function_call messages.extend",
    "iteration counter",
]

print("\nTop-3 retrieval comparison on raw vs compressed-before-index candidates:")
for q in diagnostic_queries:
    raw_top = raw_candidate_index.search(q, num_results=3)
    comp_top = compressed_candidate_index.search(q, num_results=3)
    print(f"\nquery: {q}")
    print("  raw index:       ", [(r["chunk_id"], r["filename"]) for r in raw_top])
    print("  compressed index:", [(r["chunk_id"], r["filename"]) for r in comp_top])

Optimization inflated tokens (325 -> 363); reverting to original messages


Optimization inflated tokens (473 -> 501); reverting to original messages


Exact term preservation across the retrieved chunks:
  while True           raw=3 compressed-before-index=1
  break                raw=3 compressed-before-index=0
  has_function_calls   raw=2 compressed-before-index=1
  function_call        raw=2 compressed-before-index=2
  messages.extend      raw=2 compressed-before-index=2

Top-3 retrieval comparison on raw vs compressed-before-index candidates:

query: while True break
  raw index:        [('0', '01-agentic-rag/lessons/14-agentic-loop.md'), ('1', '01-agentic-rag/lessons/14-agentic-loop.md'), ('4', '01-agentic-rag/lessons/15-frameworks.md')]
  compressed index: [('4', '01-agentic-rag/lessons/15-frameworks.md'), ('3', '01-agentic-rag/lessons/15-frameworks.md')]

query: has_function_calls False break
  raw index:        [('1', '01-agentic-rag/lessons/14-agentic-loop.md'), ('0', '01-agentic-rag/lessons/14-agentic-loop.md')]
  compressed index: [('0', '01-agentic-rag/lessons/14-agentic-loop.md'), ('1', '01-agentic-rag/lessons/14-agentic

## Scenario B — agentic search-tool outputs

In the agentic loop the model searches several times with different keywords, and each
tool output is appended to the conversation and **re-sent on every later turn**. So the
savings compound. We compress each search result and then show:

- **single-pass** savings (sum of all tool outputs once), and
- **loop-amplified** savings (output of call *k* is paid for `n_calls - k + 1` times).

In [6]:
# The keyword searches an agent might fire for the same question.
agent_queries = [
    "agentic loop while True keep calling the model",
    "function_call tool execution append messages",
    "stop condition break no function calls",
    "difference between agentic loop and plain RAG",
]

per_call = []
for q in agent_queries:
    block = render_results(chunk_index.search(q, num_results=5))
    r = compress_block(block, target_ratio=0.5)
    per_call.append((r.tokens_before, r.tokens_after))

n = len(per_call)
single_before = sum(b for b, _ in per_call)
single_after = sum(a for _, a in per_call)

# Loop amplification: call k (1-indexed) is re-sent (n - k + 1) times.
amp_before = sum(b * (n - k) for k, (b, _) in enumerate(per_call))
amp_after = sum(a * (n - k) for k, (_, a) in enumerate(per_call))

print(f"agentic searches: {n}")
print("\nper-call (before -> after):")
for k, (b, a) in enumerate(per_call, 1):
    print(f"  call {k}: {b:>5} -> {a:<5} ({1 - a / b:.0%})")

print(f"\nsingle-pass total:   {single_before} -> {single_after}  ({1 - single_after / single_before:.0%})")
print(f"loop-amplified total: {amp_before} -> {amp_after}  ({1 - amp_after / amp_before:.0%})")
print(f"  (tokens the loop stops re-sending: {amp_before - amp_after})")

agentic searches: 4

per-call (before -> after):
  call 1:  2265 -> 1984  (12%)
  call 2:  2105 -> 1746  (17%)
  call 3:  2320 -> 1820  (22%)
  call 4:  2253 -> 2181  (3%)

single-pass total:   8943 -> 7731  (14%)
loop-amplified total: 22268 -> 18995  (15%)
  (tokens the loop stops re-sending: 3273)


## End-to-end eval — is the answer preserved?

Token savings only matter if the answer survives. We ask the lab's LLM the same question
with the **full** context and the **compressed** context, and compare the answers plus the
input tokens the provider actually billed.

In [7]:
from openai import OpenAI
from rag_helper import INSTRUCTIONS, PROMPT_TEMPLATE

client = OpenAI()
LLM_MODEL = "openai/gpt-5.4-mini"  # same model as the lab (via OpenRouter)


def answer_with_context(ctx_text):
    prompt = PROMPT_TEMPLATE.format(question=query, context=ctx_text)
    resp = client.responses.create(
        model=LLM_MODEL,
        input=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt},
        ],
        max_output_tokens=1024,
    )
    return resp.output_text, resp.usage


full_answer, full_usage = answer_with_context(context)
comp_answer, comp_usage = answer_with_context(resA.messages[0]["content"])

print(f"FULL context  -> input_tokens={full_usage.input_tokens}")
print(f"COMP context  -> input_tokens={comp_usage.input_tokens}")
print(f"saved at the provider: {full_usage.input_tokens - comp_usage.input_tokens} "
      f"({1 - comp_usage.input_tokens / full_usage.input_tokens:.0%})")
print("\n=== answer from FULL context ===\n", full_answer)
print("\n=== answer from COMPRESSED context ===\n", comp_answer)

FULL context  -> input_tokens=2294
COMP context  -> input_tokens=1964
saved at the provider: 330 (14%)

=== answer from FULL context ===
 It keeps calling the model inside a `while True` loop and checks each response for function calls.

- If the model returns a `function_call`, the code runs that tool, appends the tool result to `messages`, and loops again.
- If the model returns a normal `message` with no function calls, `has_function_calls` stays `False` and the loop breaks.

So the stopping rule is: **no function calls in the current turn means the agent is done**.

=== answer from COMPRESSED context ===
 The agentic loop keeps calling the model by:

1. Sending the current `messages` history to the model.
2. Checking the model output for tool/function calls.
3. If there is a function call, executing it and appending the tool result back into `messages`.
4. Repeating the loop.

It stops when the model returns an assistant message **with no function calls**. The simplest exit conditi

## Accuracy stress test — `target_ratio=0.2`

We expected pushing `target_ratio` from `0.5` down to `0.2` to compress harder and stress
accuracy. The actual result is the more useful lesson: **for this prose+code content the
router caps the ratio and `target_ratio` makes no difference** — `0.5`, `0.2`, and even
`0.1` all yield the identical `router:mixed:0.52` (`2224 -> 1894`).

So the cell below compares `0.5` vs `0.2` and confirms:
- identical token savings (the knob is a hint the router can ignore), and
- identical answer quality — both keep `function_call` / `has_function_calls` and still
  answer correctly, even though the literal `while True` / `break` strings were already
  dropped at `0.5`.

Takeaway: you don't tune accuracy here with `target_ratio`; the meaningful levers are the
**protections** (what content is eligible at all), not the ratio.

In [8]:
key_facts = ["while True", "break", "function_call", "has_function_calls", "messages.extend"]


def fact_check(text):
    return {fact: (fact in text) for fact in key_facts}


resA_02 = compress_block(context, target_ratio=0.2)
agg_answer, agg_usage = answer_with_context(resA_02.messages[0]["content"])

print("RAG context blob compression by target_ratio")
print(f"  target_ratio=0.5: {resA.tokens_before} -> {resA.tokens_after} "
      f"({resA.compression_ratio:.0%})  {resA.transforms_applied}")
print(f"  target_ratio=0.2: {resA_02.tokens_before} -> {resA_02.tokens_after} "
      f"({resA_02.compression_ratio:.0%})  {resA_02.transforms_applied}")

print("\nProvider input tokens (end-to-end):")
print(f"  full:             {full_usage.input_tokens}")
print(f"  ratio=0.5:        {comp_usage.input_tokens} ({1 - comp_usage.input_tokens / full_usage.input_tokens:.0%} saved)")
print(f"  ratio=0.2:        {agg_usage.input_tokens} ({1 - agg_usage.input_tokens / full_usage.input_tokens:.0%} saved)")

print("\nKey-fact preservation in the compressed CONTEXT (literal substring):")
print(f"  ratio=0.5: {fact_check(resA.messages[0]['content'])}")
print(f"  ratio=0.2: {fact_check(resA_02.messages[0]['content'])}")

print("\nKey-fact coverage in the ANSWER (did the LLM still say it?):")
print(f"  full:      {fact_check(full_answer)}")
print(f"  ratio=0.2: {fact_check(agg_answer)}")

print("\n--- compressed context @0.2 (first 400 chars) ---")
print(resA_02.messages[0]["content"][:400])
print("\n=== answer from target_ratio=0.2 context ===\n", agg_answer)

RAG context blob compression by target_ratio
  target_ratio=0.5: 2224 -> 1894 (15%)  ['router:mixed:0.52']
  target_ratio=0.2: 2224 -> 1894 (15%)  ['router:mixed:0.52']

Provider input tokens (end-to-end):
  full:             2294
  ratio=0.5:        1964 (14% saved)
  ratio=0.2:        1964 (14% saved)

Key-fact preservation in the compressed CONTEXT (literal substring):
  ratio=0.5: {'while True': False, 'break': False, 'function_call': True, 'has_function_calls': True, 'messages.extend': True}
  ratio=0.2: {'while True': False, 'break': False, 'function_call': True, 'has_function_calls': True, 'messages.extend': True}

Key-fact coverage in the ANSWER (did the LLM still say it?):
  full:      {'while True': True, 'break': True, 'function_call': True, 'has_function_calls': True, 'messages.extend': False}
  ratio=0.2: {'while True': False, 'break': False, 'function_call': True, 'has_function_calls': True, 'messages.extend': False}

--- compressed context @0.2 (first 400 chars) ---
01-a

## Summary

**Practical conclusions:**

- **Index raw chunks, compress after retrieval.** `minsearch` is lexical, and pre-index compression removed exact code tokens such as `while True` / `break`, changing retrieval for code-sensitive queries.
- **Expect modest savings for this lab.** With exposed protections off, retrieved RAG context and agentic search outputs saved about **14-15%**, not 60-95%. That is aligned with Headroom's own benchmarks for short, dense, code-heavy content.
- **The meaningful levers are protections, not `target_ratio`.** In this corpus, `target_ratio=0.5`, `0.2`, and `0.1` all converged to the same `router:mixed:0.52` output. Turning code/user protections on/off mattered much more.
- **Validate answer quality, especially for code-detail questions.** Compression kept enough signal for this question, but literal code strings disappeared from the compressed context.
- **Keep Headroom as an optional optimization layer.** The lab should still work with uncompressed context; compression should be measurable and easy to bypass.

**CCR note:** In this notebook we use plain library mode: `compress()` followed by a direct `OpenAI()` call. CCR markers may appear in compressed context (for example, `Retrieve more: hash=...`), but automatic recovery is **not enabled** because the model does not receive a `headroom_retrieve` tool. To make compression reversible in practice, use Headroom proxy, `HeadroomClient`, or MCP retrieval; otherwise rely on evals to catch dropped details.

In [9]:
rows = [
    ("A. RAG context blob (Kompress)", resA.tokens_before, resA.tokens_after),
    ("B. agentic outputs (single-pass)", single_before, single_after),
    ("B. agentic outputs (loop-amplified)", amp_before, amp_after),
    ("A. end-to-end provider input tokens", full_usage.input_tokens, comp_usage.input_tokens),
]

print(f"{'scenario':38} {'before':>8} {'after':>8} {'saved':>8}")
print("-" * 66)
for name, b, a in rows:
    print(f"{name:38} {b:>8} {a:>8} {1 - a / b:>7.0%}")

scenario                                 before    after    saved
------------------------------------------------------------------
A. RAG context blob (Kompress)             2224     1894     15%
B. agentic outputs (single-pass)           8943     7731     14%
B. agentic outputs (loop-amplified)       22268    18995     15%
A. end-to-end provider input tokens        2294     1964     14%
